# Explainable Fraud Alert Prioritization - Data Exploration

## Problem Statement
Financial institutions receive high volumes of fraud alerts. Analysts need a way to prioritise alerts, understand the strongest risk drivers, and explain why a transaction should be reviewed.

## Progress Report / Approach
This notebook documents the first stage of the project:
- understand the raw customer, merchant, and transaction datasets
- validate data quality before modelling
- check missing values, duplicate rows, and data types
- explore fraud distribution and risk patterns
- identify early indicators for feature engineering and model design


In [ ]:
# Import the core libraries used for data analysis and visualisation.
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make notebook tables easier to inspect during review.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# Add the project src folder so notebook code reuses production pipeline logic.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(REPO_ROOT / 'src'))


## Dataset Loading
The project uses three raw datasets: customers, merchants, and transactions. Transactions contain the target label used for fraud modelling.


In [ ]:
# Load all raw datasets from the project data folder.
# The fallback keeps the notebook working if the older data/raw/raw layout exists.
def load_csv(name):
    clean_path = REPO_ROOT / 'data' / 'raw' / name
    nested_path = REPO_ROOT / 'data' / 'raw' / 'raw' / name
    path = clean_path if clean_path.exists() else nested_path
    return pd.read_csv(path)

customers = load_csv('customers.csv')
merchants = load_csv('merchants.csv')
transactions = load_csv('transactions.csv')

print('Customers shape:', customers.shape)
print('Merchants shape:', merchants.shape)
print('Transactions shape:', transactions.shape)


## Dataset Overview
This step confirms that the files loaded correctly and gives reviewers a quick view of each dataset.


In [ ]:
# Display a small sample from each dataset to validate the loaded schema.
display(customers.head())
display(merchants.head())
display(transactions.head())


## Data Cleaning Summary
The datasets were inspected for common data quality issues before modelling.

### Missing Values
A complete missing-value analysis was performed on customer, merchant, and transaction datasets.

### Duplicate Records
Duplicate checks were performed on all datasets.

### Data Quality Assessment
If missing values or duplicates are found, they should be handled before feature engineering. In the current dataset, the checks below show that the raw data is already clean enough to proceed.


In [ ]:
# Missing-value check for each raw dataset.
print('CUSTOMERS MISSING VALUES')
display(customers.isnull().sum())

print('MERCHANTS MISSING VALUES')
display(merchants.isnull().sum())

print('TRANSACTIONS MISSING VALUES')
display(transactions.isnull().sum())


In [ ]:
# Duplicate-row check for each raw dataset.
print('Customer duplicates:', customers.duplicated().sum())
print('Merchant duplicates:', merchants.duplicated().sum())
print('Transaction duplicates:', transactions.duplicated().sum())


## Data Type Validation
The next step is to validate field types before modelling. Dates, numeric risk scores, binary flags, and categorical fields need to be treated correctly.


In [ ]:
# Validate inferred data types.
print('CUSTOMERS DATA TYPES')
display(customers.dtypes)

print('MERCHANTS DATA TYPES')
display(merchants.dtypes)

print('TRANSACTIONS DATA TYPES')
display(transactions.dtypes)


In [ ]:
# Convert transaction timestamps to datetime for time-based analysis.
transactions['event_ts'] = pd.to_datetime(transactions['event_ts'], errors='coerce')

# Confirm whether timestamp conversion introduced any invalid values.
print('Invalid transaction timestamps:', transactions['event_ts'].isna().sum())


## Fraud Distribution
Fraud datasets are usually imbalanced because genuine transactions are much more common than fraudulent ones. This affects model selection and evaluation.


In [ ]:
# Check class balance for the target variable.
fraud_counts = transactions['fraud_label'].value_counts().sort_index()
fraud_rate = transactions['fraud_label'].mean()

print(fraud_counts)
print(f'Fraud rate: {fraud_rate:.2%}')

plt.figure(figsize=(6, 4))
sns.countplot(data=transactions, x='fraud_label')
plt.title('Fraud vs Legitimate Transactions')
plt.xlabel('Fraud Label')
plt.ylabel('Transaction Count')
plt.show()


## Exploratory Risk Analysis
The following charts inspect transaction amount, device risk, merchant risk, velocity, and geography. These are common fraud investigation drivers.


In [ ]:
# Summarise the main numeric risk fields.
risk_features = [
    'transaction_amount_usd',
    'device_risk_score',
    'merchant_risk_score',
    'velocity_1h',
    'velocity_24h',
    'geo_distance_km',
]

display(transactions[risk_features].describe())


In [ ]:
# Compare transaction amount distributions for legitimate and fraudulent activity.
plt.figure(figsize=(10, 5))
sns.boxplot(data=transactions, x='fraud_label', y='transaction_amount_usd')
plt.title('Transaction Amount by Fraud Label')
plt.xlabel('Fraud Label')
plt.ylabel('Transaction Amount USD')
plt.show()


In [ ]:
# Review correlation between numeric fields and the fraud label.
numeric_cols = transactions.select_dtypes(include=np.number)
correlation_matrix = numeric_cols.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, cmap='Blues', annot=False)
plt.title('Numeric Feature Correlation Matrix')
plt.show()

display(correlation_matrix['fraud_label'].sort_values(ascending=False))


## Initial Fraud Risk Observations
The strongest early fraud indicators are expected to come from device risk, merchant risk, transaction velocity, geographic distance, and night-time behaviour. These findings guide the feature engineering and top-feature selection notebooks.
